In [3]:
# =============================================================
# STEP 1 — CELL 1
# Import libraries and verify input file paths
# =============================================================

import pandas as pd
from pathlib import Path

# ---- INPUT FILE PATHS ----
ERA5_PATH = Path(r"C:\Users\n.mohammadi\Desktop\NimaMohammadi\02.Nima Mohammadi - Thesis\PythonProjects\ClimateProcessingProject\data\ERA5_clean\ERA5_Milan_1990_2024_daily.csv")

BUILDINGS_PATH = Path(r"C:\Users\n.mohammadi\Desktop\NimaMohammadi\02.Nima Mohammadi - Thesis\PythonProjects\CENED_Project\data\01_processed\spatial_join_v2\run_20260222_171925\buildings_with_epc_v2.csv")

# ---- OUTPUT FOLDERS ----
PROJECT_ROOT = Path(r"C:\Users\n.mohammadi\Desktop\NimaMohammadi\02.Nima Mohammadi - Thesis\ML Dataset")
INTERMEDIATE_DIR = PROJECT_ROOT / "03_intermediate"
FINAL_DIR = PROJECT_ROOT / "04_final_ml_dataset"
LOGS_DIR = PROJECT_ROOT / "05_logs"

# ---- VERIFY SOURCE FILES EXIST ----
print("Checking input files...\n")

if ERA5_PATH.exists():
    size_mb = ERA5_PATH.stat().st_size / (1024 * 1024)
    print(f"  [OK] ERA5 file found     ({size_mb:.2f} MB)")
else:
    print(f"  [MISSING] ERA5 file NOT found at:\n             {ERA5_PATH}")

if BUILDINGS_PATH.exists():
    size_mb = BUILDINGS_PATH.stat().st_size / (1024 * 1024)
    print(f"  [OK] Buildings file found ({size_mb:.2f} MB)")
else:
    print(f"  [MISSING] Buildings file NOT found at:\n             {BUILDINGS_PATH}")

# ---- VERIFY OUTPUT FOLDERS EXIST ----
print("\nChecking output folders...\n")
for folder in [INTERMEDIATE_DIR, FINAL_DIR, LOGS_DIR]:
    if folder.exists():
        print(f"  [OK] {folder.name}")
    else:
        print(f"  [MISSING] {folder.name}")

print("\nDone.")

Checking input files...

  [OK] ERA5 file found     (4.16 MB)
  [OK] Buildings file found (31.02 MB)

Checking output folders...

  [OK] 03_intermediate
  [OK] 04_final_ml_dataset
  [OK] 05_logs

Done.


In [4]:
# =============================================================
# STEP 1 — CELL 2
# Load ERA5 climate data and verify its content
# =============================================================

# Load ERA5 daily climate data
print("Loading ERA5 climate data...\n")
era5 = pd.read_csv(ERA5_PATH)

# Convert 'time' column to datetime format
era5['time'] = pd.to_datetime(era5['time'])

# ---- BASIC INFORMATION ----
print(f"Shape: {era5.shape[0]:,} rows  ×  {era5.shape[1]} columns\n")

print(f"Date range:")
print(f"  From: {era5['time'].min().date()}")
print(f"  To:   {era5['time'].max().date()}")

n_years = era5['time'].dt.year.nunique()
print(f"  Years covered: {n_years}\n")

# ---- COLUMN NAMES ----
print("Columns in the dataset:")
for i, col in enumerate(era5.columns, 1):
    print(f"  {i:2d}. {col}")

# ---- FIRST FEW ROWS ----
print("\nFirst 5 rows (preview):")
era5.head()

Loading ERA5 climate data...

Shape: 12,784 rows  ×  19 columns

Date range:
  From: 1990-01-01
  To:   2024-12-31
  Years covered: 35

Columns in the dataset:
   1. time
   2. dewpoint_temperature_2m
   3. evaporation_from_bare_soil
   4. evaporation_from_open_water_surfaces_excluding_oceans
   5. potential_evaporation
   6. skin_temperature
   7. surface_net_solar_radiation
   8. surface_net_thermal_radiation
   9. surface_pressure
  10. surface_solar_radiation_downwards
  11. surface_thermal_radiation_downwards
  12. temperature_2m
  13. total_evaporation
  14. total_precipitation
  15. u_component_of_wind_10m
  16. v_component_of_wind_10m
  17. volumetric_soil_water_layer_1
  18. wind_speed_10m
  19. year

First 5 rows (preview):


,time,dewpoint_temperature_2m,evaporation_from_bare_soil,evaporation_from_open_water_surfaces_excluding_oceans,potential_evaporation,skin_temperature,surface_net_solar_radiation,surface_net_thermal_radiation,surface_pressure,surface_solar_radiation_downwards,surface_thermal_radiation_downwards,temperature_2m,total_evaporation,total_precipitation,u_component_of_wind_10m,v_component_of_wind_10m,volumetric_soil_water_layer_1,wind_speed_10m,year
0,1990-01-01,-5.056007,-0.000002,-0.000013,-0.000309,-2.650599,693.907724,-683.822209,100014.817097,802.644514,3076.099459,-3.189028,-0.000021,0.003621,0.998954,-0.258948,0.430315,1.145791,1990
1,1990-01-02,-4.943282,-0.000002,-0.000011,-0.000315,-2.590518,776.261460,-780.820247,99882.806251,897.531968,2978.141673,-2.539750,-0.000021,0.002283,0.786257,-0.345120,0.429229,1.023357,1990
2,1990-01-03,-4.455809,-0.000003,-0.000011,-0.000303,-1.993348,822.563560,-808.627321,99968.854107,950.753098,2977.127017,-1.654876,-0.000019,0.000487,-0.111026,-0.500728,0.428195,1.012839,1990
3,1990-01-04,-2.678010,-0.000004,-0.000012,-0.000240,-1.117184,636.098449,-529.449444,100598.551367,734.760057,3332.996628,-0.925320,-0.000022,0.000744,0.562771,-0.712550,0.427213,1.293427,1990
4,1990-01-05,-4.062554,-0.000004,-0.000013,-0.000278,-2.345535,736.553352,-694.557060,100724.552069,850.630783,3098.511624,-2.218500,-0.000025,0.003279,0.127955,-0.194423,0.426276,0.994992,1990


In [5]:
# =============================================================
# STEP 1 — CELL 3
# Compute daily Relative Humidity and Heat Index
# =============================================================

import numpy as np

print("Computing derived climate variables...\n")

# ---- 1. RELATIVE HUMIDITY (Magnus formula) ----
# Standard meteorological formula using temperature and dewpoint
# RH = 100 * exp((a * Td)/(b + Td)) / exp((a * T)/(b + T))
a, b = 17.625, 243.04
T  = era5['temperature_2m']
Td = era5['dewpoint_temperature_2m']

era5['relative_humidity'] = 100.0 * (
    np.exp((a * Td) / (b + Td)) /
    np.exp((a * T)  / (b + T))
)

# Clip to physical bounds [0, 100]
era5['relative_humidity'] = era5['relative_humidity'].clip(0, 100)

# ---- 2. HEAT INDEX (Rothfusz regression, NOAA) ----
# Heat Index requires temperatures in Fahrenheit; we convert, compute, convert back
T_F  = T * 9/5 + 32
RH   = era5['relative_humidity']

# Full Rothfusz formula (used when T_F >= 80°F, i.e. ~26.7°C)
HI_F = (-42.379
        + 2.04901523 * T_F
        + 10.14333127 * RH
        - 0.22475541 * T_F * RH
        - 0.00683783 * T_F**2
        - 0.05481717 * RH**2
        + 0.00122874 * T_F**2 * RH
        + 0.00085282 * T_F * RH**2
        - 0.00000199 * T_F**2 * RH**2)

# For cool conditions, HI is essentially the air temperature itself
HI_F_cool = 0.5 * (T_F + 61.0 + ((T_F - 68.0) * 1.2) + (RH * 0.094))

# Combine: use cool formula when T_F < 80, otherwise use full formula
HI_F_final = np.where(T_F < 80, HI_F_cool, HI_F)

# Convert back to Celsius
era5['heat_index'] = (HI_F_final - 32) * 5/9

# ---- 3. SUMMARY ----
print("Relative Humidity (%):")
print(f"  min:  {era5['relative_humidity'].min():.1f}")
print(f"  mean: {era5['relative_humidity'].mean():.1f}")
print(f"  max:  {era5['relative_humidity'].max():.1f}")

print("\nHeat Index (°C):")
print(f"  min:  {era5['heat_index'].min():.2f}")
print(f"  mean: {era5['heat_index'].mean():.2f}")
print(f"  max:  {era5['heat_index'].max():.2f}")

print("\nDone.")

Computing derived climate variables...

Relative Humidity (%):
  min:  20.7
  mean: 72.3
  max:  98.4

Heat Index (°C):
  min:  -9.06
  mean: 12.38
  max:  34.50

Done.


In [6]:
# =============================================================
# STEP 1 — CELL 4
# Compute the 8 long-term climate indices for Milan (1990–2024)
# =============================================================

print("Computing the 8 climate indices...\n")

# Add helper columns for grouping
era5['year']  = era5['time'].dt.year
era5['month'] = era5['time'].dt.month

# ---- 1. CDD26 — Cooling Degree Days base 26°C ----
# For each day, CDD = max(0, T - 26). Then sum per year, then average across years.
daily_cdd = (era5['temperature_2m'] - 26).clip(lower=0)
annual_cdd = daily_cdd.groupby(era5['year']).sum()
CDD26 = annual_cdd.mean()

# ---- 2. mean_temp — Long-term annual mean temperature ----
mean_temp = era5['temperature_2m'].mean()

# ---- 3. mean_temp_summer — JJA (June-July-August) mean temperature ----
summer_mask = era5['month'].isin([6, 7, 8])
mean_temp_summer = era5.loc[summer_mask, 'temperature_2m'].mean()

# ---- 4. max_temp — Long-term annual mean of yearly maximums ----
annual_max = era5.groupby('year')['temperature_2m'].max()
max_temp = annual_max.mean()

# ---- 5. RH_mean — Long-term annual mean relative humidity ----
RH_mean = era5['relative_humidity'].mean()

# ---- 6. HI_mean — Long-term annual mean heat index ----
HI_mean = era5['heat_index'].mean()

# ---- 7. ts_anom — Temperature anomaly vs 1991–2020 baseline ----
baseline_mask = (era5['year'] >= 1991) & (era5['year'] <= 2020)
baseline_mean = era5.loc[baseline_mask, 'temperature_2m'].mean()
ts_anom = mean_temp - baseline_mean

# ---- 8. ts_anom_summer — Summer temperature anomaly vs 1991–2020 baseline ----
baseline_summer_mask = baseline_mask & summer_mask
baseline_summer_mean = era5.loc[baseline_summer_mask, 'temperature_2m'].mean()
ts_anom_summer = mean_temp_summer - baseline_summer_mean

# ---- DISPLAY RESULTS ----
indices = {
    'CDD26':            (CDD26,            'cooling degree-days/year'),
    'mean_temp':        (mean_temp,        '°C'),
    'mean_temp_summer': (mean_temp_summer, '°C'),
    'max_temp':         (max_temp,         '°C'),
    'RH_mean':          (RH_mean,          '%'),
    'HI_mean':          (HI_mean,          '°C'),
    'ts_anom':          (ts_anom,          '°C'),
    'ts_anom_summer':   (ts_anom_summer,   '°C'),
}

print(f"{'Index':<20} {'Value':>12}   Unit")
print("-" * 55)
for name, (value, unit) in indices.items():
    print(f"{name:<20} {value:>12.4f}   {unit}")

# ---- SAVE TO INTERMEDIATE FOLDER ----
indices_df = pd.DataFrame({
    'index_name': list(indices.keys()),
    'value':      [v[0] for v in indices.values()],
    'unit':       [v[1] for v in indices.values()],
})

output_path = INTERMEDIATE_DIR / "climate_indices_milan_1990_2024.csv"
indices_df.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path.name}")
print("\nDone.")

Computing the 8 climate indices...

Index                       Value   Unit
-------------------------------------------------------
CDD26                     22.3989   cooling degree-days/year
mean_temp                 13.1106   °C
mean_temp_summer          22.9264   °C
max_temp                  28.6118   °C
RH_mean                   72.3438   %
HI_mean                   12.3810   °C
ts_anom                    0.1001   °C
ts_anom_summer             0.1150   °C

Saved to: climate_indices_milan_1990_2024.csv

Done.


In [7]:
# =============================================================
# STEP 2 — CELL 5
# Load the buildings dataset and verify its content
# =============================================================

print("Loading buildings dataset...\n")

# Load the buildings file (low_memory=False avoids dtype warnings on large mixed-type files)
buildings = pd.read_csv(BUILDINGS_PATH, low_memory=False)

# ---- BASIC INFORMATION ----
print(f"Shape: {buildings.shape[0]:,} rows  ×  {buildings.shape[1]} columns\n")

# ---- TARGET VARIABLE: CLIMATIZZAZIONE_ESTIVA (cooling presence) ----
print("Target variable (CLIMATIZZAZIONE_ESTIVA):")
target_counts = buildings['CLIMATIZZAZIONE_ESTIVA'].value_counts(dropna=False)
total = len(buildings)
for value, count in target_counts.items():
    pct = count / total * 100
    label = "Cooling YES" if value is True else ("Cooling NO" if value is False else "No EPC label")
    print(f"  {label:<15}: {count:>7,}  ({pct:5.1f}%)")

# ---- VERIFY KEY COLUMNS ----
print("\nKey columns check:")
key_cols = ['FEATURE_ID', 'EDIFC_TY', 'EDIFC_USO', 'CLIMATIZZAZIONE_ESTIVA',
            'ANNO_COSTRUZIONE', 'SUPERF_UTILE_RISCALDATA']
for col in key_cols:
    if col in buildings.columns:
        print(f"  [OK]      {col}")
    else:
        print(f"  [MISSING] {col}")

print("\nDone.")

Loading buildings dataset...

Shape: 53,041 rows  ×  218 columns

Target variable (CLIMATIZZAZIONE_ESTIVA):
  No EPC label   :  33,978  ( 64.1%)
  Cooling YES    :  13,787  ( 26.0%)
  Cooling NO     :   5,276  (  9.9%)

Key columns check:
  [OK]      FEATURE_ID
  [OK]      EDIFC_TY
  [OK]      EDIFC_USO
  [OK]      CLIMATIZZAZIONE_ESTIVA
  [OK]      ANNO_COSTRUZIONE
  [OK]      SUPERF_UTILE_RISCALDATA

Done.


In [8]:
# =============================================================
# STEP 2 — CELL 6
# Add the 8 climate indices to all buildings and save the merged file
# =============================================================

print("Adding climate indices to all buildings...\n")

# ---- ATTACH EACH CLIMATE INDEX AS A NEW COLUMN ----
# Same value for every building (city-scale ERA5 data)
buildings['CDD26']            = CDD26
buildings['mean_temp']        = mean_temp
buildings['mean_temp_summer'] = mean_temp_summer
buildings['max_temp']         = max_temp
buildings['RH_mean']          = RH_mean
buildings['HI_mean']          = HI_mean
buildings['ts_anom']          = ts_anom
buildings['ts_anom_summer']   = ts_anom_summer

# ---- VERIFY THE MERGE ----
print(f"New shape: {buildings.shape[0]:,} rows  ×  {buildings.shape[1]} columns")
print(f"  (was 53,041 × 218, now should be 53,041 × 226)\n")

# Show the 8 new columns and their values
print("Climate features attached to every building:")
print("-" * 55)
climate_cols = ['CDD26', 'mean_temp', 'mean_temp_summer', 'max_temp',
                'RH_mean', 'HI_mean', 'ts_anom', 'ts_anom_summer']
for col in climate_cols:
    val = buildings[col].iloc[0]   # all buildings have the same value
    nuniq = buildings[col].nunique()
    print(f"  {col:<20}  value = {val:>10.4f}   unique values = {nuniq}")

# ---- SAVE TO INTERMEDIATE FOLDER ----
output_path = INTERMEDIATE_DIR / "buildings_with_epc_and_climate_v1.csv"
print(f"\nSaving to: {output_path.name}")
print("(This may take 30-60 seconds for a 32 MB file...)")

buildings.to_csv(output_path, index=False)

# ---- CONFIRM FILE WAS SAVED ----
size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"\n[OK] File saved successfully ({size_mb:.2f} MB)")
print(f"     Location: 03_intermediate/{output_path.name}")

print("\nDone.")

Adding climate indices to all buildings...

New shape: 53,041 rows  ×  226 columns
  (was 53,041 × 218, now should be 53,041 × 226)

Climate features attached to every building:
-------------------------------------------------------
  CDD26                 value =    22.3989   unique values = 1
  mean_temp             value =    13.1106   unique values = 1
  mean_temp_summer      value =    22.9264   unique values = 1
  max_temp              value =    28.6118   unique values = 1
  RH_mean               value =    72.3438   unique values = 1
  HI_mean               value =    12.3810   unique values = 1
  ts_anom               value =     0.1001   unique values = 1
  ts_anom_summer        value =     0.1150   unique values = 1

Saving to: buildings_with_epc_and_climate_v1.csv
(This may take 30-60 seconds for a 32 MB file...)


C:\Users\n.mohammadi\AppData\Local\Temp\ipykernel_17536\2289318206.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  buildings['CDD26']            = CDD26
C:\Users\n.mohammadi\AppData\Local\Temp\ipykernel_17536\2289318206.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  buildings['mean_temp']        = mean_temp
C:\Users\n.mohammadi\AppData\Local\Temp\ipykernel_17536\2289318206.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform


[OK] File saved successfully (37.78 MB)
     Location: 03_intermediate/buildings_with_epc_and_climate_v1.csv

Done.


In [9]:
# =============================================================
# STEP 3 — CELL 7
# Filter to labeled buildings only (Stage 1 training set)
# =============================================================

print("Filtering to buildings with EPC labels (Stage 1 ML set)...\n")

# ---- KEEP ONLY ROWS WHERE THE TARGET IS NOT NULL ----
labeled = buildings[buildings['CLIMATIZZAZIONE_ESTIVA'].notna()].copy()

# ---- REPORT BEFORE / AFTER ----
print(f"Before filter:  {len(buildings):>7,} buildings  (full DBT inventory)")
print(f"After filter:   {len(labeled):>7,} buildings  (with EPC label)")
print(f"Set aside:      {len(buildings) - len(labeled):>7,} buildings  (no EPC, kept for later prediction)")

# ---- CLASS BALANCE OF THE TARGET ----
print("\nClass balance of target (CLIMATIZZAZIONE_ESTIVA):")
target_counts = labeled['CLIMATIZZAZIONE_ESTIVA'].value_counts()
n_total = len(labeled)
for value, count in target_counts.items():
    pct = count / n_total * 100
    label = "Cooling YES (positive class)" if value is True else "Cooling NO  (negative class)"
    print(f"  {label:<32}: {count:>6,}  ({pct:5.1f}%)")

# ---- IMBALANCE RATIO ----
yes_count = (labeled['CLIMATIZZAZIONE_ESTIVA'] == True).sum()
no_count  = (labeled['CLIMATIZZAZIONE_ESTIVA'] == False).sum()
ratio = yes_count / no_count
print(f"\nImbalance ratio (YES : NO) = {ratio:.2f} : 1")
if ratio > 2 or ratio < 0.5:
    print("  -> Note: classes are moderately imbalanced. We will handle this during model training.")
else:
    print("  -> Classes are reasonably balanced.")

print(f"\nLabeled dataset shape: {labeled.shape[0]:,} rows  ×  {labeled.shape[1]} columns")
print("\nDone.")

Filtering to buildings with EPC labels (Stage 1 ML set)...

Before filter:   53,041 buildings  (full DBT inventory)
After filter:    19,063 buildings  (with EPC label)
Set aside:       33,978 buildings  (no EPC, kept for later prediction)

Class balance of target (CLIMATIZZAZIONE_ESTIVA):
  Cooling YES (positive class)    : 13,787  ( 72.3%)
  Cooling NO  (negative class)    :  5,276  ( 27.7%)

Imbalance ratio (YES : NO) = 2.61 : 1
  -> Note: classes are moderately imbalanced. We will handle this during model training.

Labeled dataset shape: 19,063 rows  ×  226 columns

Done.


In [10]:
# =============================================================
# STEP 3 — CELL 8
# Identify columns to drop (diagnostic only — no changes yet)
# =============================================================

print("Auditing all 226 columns in the labeled dataset...\n")

# ---- GROUP 1: 100% empty columns ----
missing_pct = labeled.isnull().mean() * 100
empty_cols = missing_pct[missing_pct == 100].index.tolist()

# ---- GROUP 2: constant columns (only one unique non-null value) ----
constant_cols = []
for col in labeled.columns:
    n_unique = labeled[col].dropna().nunique()
    if n_unique <= 1 and col not in empty_cols:
        constant_cols.append(col)

# ---- GROUP 3: leakage columns ----
# Any column whose name contains keywords directly tied to the cooling system.
# These would let the model "cheat" by directly reading the answer.
leakage_keywords = [
    'RAFFRESC',                    # any cooling-related column (cooled area, cooled volume, cooling consumption, etc.)
    'CLIMATIZZAZIONE_ESTIVA',      # the target itself and any sibling field
    'CE_',                         # cooling energy index columns (CE_EPREN, etc.)
    'EP_C',                        # EP_C* are cooling-specific energy performance indices
    'A_SOL_EST_A_SUP_UTILE',       # solar gains in summer — strongly tied to cooling system design
]
leakage_cols = []
for col in labeled.columns:
    for kw in leakage_keywords:
        if kw in col:
            leakage_cols.append(col)
            break

# Make sure we don't list the target itself as something to "drop" — we keep it as y
if 'CLIMATIZZAZIONE_ESTIVA' in leakage_cols:
    leakage_cols.remove('CLIMATIZZAZIONE_ESTIVA')

# ---- GROUP 4: identifier / non-feature columns ----
# These are used for tracking, not for prediction.
identifier_cols = [
    'FEATURE_ID',          # building unique ID
    'COD_APE',             # EPC certificate ID
    'INDIRIZZO',           # raw address
    'INDIRIZZO_NORM',      # normalized address
    'COMUNE',              # always "MILANO"
    'PROVINCIA',           # always "MI"
    'REGIONE',             # always "Lombardia"
    'DATA_INS',            # date of certificate
    'match_method',        # how the spatial join matched the EPC to the building
    'nearest_dist_m',      # distance of nearest-join
]
# Keep only those that actually exist in the dataset
identifier_cols = [c for c in identifier_cols if c in labeled.columns]

# ---- DEDUPLICATE ACROSS GROUPS ----
# Some columns may appear in multiple groups; we want each listed once.
all_to_drop = set()
for group in [empty_cols, constant_cols, leakage_cols, identifier_cols]:
    all_to_drop.update(group)

# ---- REPORT ----
print(f"GROUP 1 — Empty columns (100% missing):     {len(empty_cols):>3} columns")
print(f"GROUP 2 — Constant columns (1 unique val):  {len(constant_cols):>3} columns")
print(f"GROUP 3 — Leakage columns (cooling-related):{len(leakage_cols):>3} columns")
print(f"GROUP 4 — Identifier / non-feature columns: {len(identifier_cols):>3} columns")
print(f"{'─'*55}")
print(f"TOTAL unique columns to drop:               {len(all_to_drop):>3} columns")
print(f"Columns remaining after drop:               {labeled.shape[1] - len(all_to_drop):>3} columns")

# ---- DETAILED LISTS ----
print("\n" + "="*60)
print("DETAILED LISTS (please review carefully):")
print("="*60)

print(f"\n[GROUP 3] LEAKAGE columns ({len(leakage_cols)}):  ← MOST IMPORTANT to verify")
for c in sorted(leakage_cols):
    print(f"    {c}")

print(f"\n[GROUP 4] IDENTIFIER columns ({len(identifier_cols)}):")
for c in sorted(identifier_cols):
    print(f"    {c}")

print(f"\n[GROUP 2] CONSTANT columns ({len(constant_cols)}):")
for c in sorted(constant_cols):
    sample_val = labeled[c].dropna().iloc[0] if labeled[c].notna().any() else "all-null"
    print(f"    {c}   (always = {sample_val})")

print(f"\n[GROUP 1] EMPTY columns ({len(empty_cols)}):")
# Print only first 10 + count, since there are many
for c in sorted(empty_cols)[:10]:
    print(f"    {c}")
if len(empty_cols) > 10:
    print(f"    ... and {len(empty_cols) - 10} more")

# ---- STORE THE LIST FOR CELL 9 ----
columns_to_drop = sorted(all_to_drop)

print(f"\n[OK] List of {len(columns_to_drop)} columns prepared for dropping in Cell 9.")
print("Done.")

Auditing all 226 columns in the labeled dataset...

GROUP 1 — Empty columns (100% missing):      28 columns
GROUP 2 — Constant columns (1 unique val):   45 columns
GROUP 3 — Leakage columns (cooling-related): 23 columns
GROUP 4 — Identifier / non-feature columns:   9 columns
───────────────────────────────────────────────────────
TOTAL unique columns to drop:                94 columns
Columns remaining after drop:               132 columns

DETAILED LISTS (please review carefully):

[GROUP 3] LEAKAGE columns (23):  ← MOST IMPORTANT to verify
    A_SOL_EST_A_SUP_UTILE
    CE_ANNO_INSTALLAZIONE_1
    CE_ANNO_INSTALLAZIONE_2
    CE_EFFICIENZA_MEDIA
    CE_EPNREN
    CE_EPREN
    CE_POTENZA_NOMINALE_1
    CE_POTENZA_NOMINALE_2
    CE_TIPO_IMPIANTO_1
    CE_TIPO_IMPIANTO_2
    CE_VETTORE_ENERGETICO_1
    CE_VETTORE_ENERGETICO_2
    CI_CODICE_CATASTO_1
    CONSUMI_TELERAFFRESCAMENTO
    DS_CODICE_INTERVENTO_1
    DS_CODICE_INTERVENTO_2
    DS_CODICE_INTERVENTO_3
    DS_CODICE_INTERVENTO_4
  

In [11]:
# =============================================================
# STEP 3 — CELL 8b (UPDATED — based on EPC leakage evidence)
# Refine drop list: protect climate features, save A_SOL_EST_A_SUP_UTILE
# =============================================================

print("Refining the drop list with evidence from EPC leakage analysis...\n")

# ---- 1. PROTECT THE 8 CLIMATE FEATURES ----
# They look "constant" today but are essential for Stage 2 future-scenario predictions.
climate_features = [
    'CDD26', 'mean_temp', 'mean_temp_summer', 'max_temp',
    'RH_mean', 'HI_mean', 'ts_anom', 'ts_anom_summer'
]
removed_climate = []
for c in climate_features:
    if c in columns_to_drop:
        columns_to_drop.remove(c)
        removed_climate.append(c)
print(f"PROTECTED {len(removed_climate)} climate features (essential for Stage 2):")
for c in removed_climate:
    print(f"  KEEP -> {c}")

# ---- 2. PROTECT A_SOL_EST_A_SUP_UTILE ----
# Verified: this is a building DESIGN parameter (summer solar gain / useful surface),
# NOT a cooling-system parameter. It is safe to use as a predictor.
saved_design = []
if 'A_SOL_EST_A_SUP_UTILE' in columns_to_drop:
    columns_to_drop.remove('A_SOL_EST_A_SUP_UTILE')
    saved_design.append('A_SOL_EST_A_SUP_UTILE')
print(f"\nPROTECTED {len(saved_design)} building design feature(s):")
for c in saved_design:
    print(f"  KEEP -> {c}")

# ---- 3. CONFIRM REMAINING LEAKAGE COLUMNS (verified from EPC analysis) ----
# These remain on the drop list because they are confirmed leakage:
#   - SUPERF_UTILE_RAFFRESCATA: cooled surface (perfect leakage)
#   - VOLUME_LORDO_RAFFRESCATO: cooled volume (perfect leakage)
#   - CONSUMI_TELERAFFRESCAMENTO: district cooling consumption (always 0 in Milan)
#   - All CE_* columns: cooling system specifications (leakage)
#   - All CI_* columns: cooling system identifiers (leakage)
#   - DS_CODICE_INTERVENTO_*: constant columns, no info

# ---- 4. FINAL REVIEW ----
print(f"\n{'='*60}")
print(f"FINAL drop list: {len(columns_to_drop)} columns")
print(f"Columns to KEEP after drop: {labeled.shape[1] - len(columns_to_drop)}")
print(f"{'='*60}")

# Quick sanity check: list a few important columns we are KEEPING
critical_keeps = ['CLIMATIZZAZIONE_ESTIVA',  # the target
                  'EDIFC_TY', 'EDIFC_USO',   # DBT geometry
                  'ANNO_COSTRUZIONE',         # building age
                  'SUPERF_UTILE_RISCALDATA',  # heated area (NOT cooled)
                  'CDD26', 'mean_temp_summer',  # climate
                  'A_SOL_EST_A_SUP_UTILE']    # design parameter
print("\nSanity check — these critical columns must be KEPT (not dropped):")
for c in critical_keeps:
    if c in labeled.columns:
        status = "WILL BE DROPPED ⚠️" if c in columns_to_drop else "kept ✓"
        print(f"  {c:<32} {status}")

print("\nDone.")

Refining the drop list with evidence from EPC leakage analysis...

PROTECTED 8 climate features (essential for Stage 2):
  KEEP -> CDD26
  KEEP -> mean_temp
  KEEP -> mean_temp_summer
  KEEP -> max_temp
  KEEP -> RH_mean
  KEEP -> HI_mean
  KEEP -> ts_anom
  KEEP -> ts_anom_summer

PROTECTED 1 building design feature(s):
  KEEP -> A_SOL_EST_A_SUP_UTILE

FINAL drop list: 85 columns
Columns to KEEP after drop: 141

Sanity check — these critical columns must be KEPT (not dropped):
  CLIMATIZZAZIONE_ESTIVA           kept ✓
  EDIFC_TY                         kept ✓
  EDIFC_USO                        kept ✓
  ANNO_COSTRUZIONE                 kept ✓
  SUPERF_UTILE_RISCALDATA          kept ✓
  CDD26                            kept ✓
  mean_temp_summer                 kept ✓
  A_SOL_EST_A_SUP_UTILE            kept ✓

Done.


In [12]:
# =============================================================
# STEP 3 — CELL 9
# Drop the 85 verified columns (leakage, empty, constant, identifiers)
# =============================================================

print(f"Dropping {len(columns_to_drop)} columns from the labeled dataset...\n")

# ---- BEFORE / AFTER ----
shape_before = labeled.shape
print(f"Shape BEFORE drop: {shape_before[0]:,} rows  ×  {shape_before[1]} columns")

# ---- ACTUAL DROP ----
ml_dataset = labeled.drop(columns=columns_to_drop)

shape_after = ml_dataset.shape
print(f"Shape AFTER drop:  {shape_after[0]:,} rows  ×  {shape_after[1]} columns")
print(f"Columns removed:   {shape_before[1] - shape_after[1]}")

# ---- VERIFY THE TARGET IS STILL PRESENT ----
if 'CLIMATIZZAZIONE_ESTIVA' in ml_dataset.columns:
    print("\n[OK] Target variable CLIMATIZZAZIONE_ESTIVA is preserved.")
else:
    print("\n[ERROR] Target variable was lost! Stop and investigate.")

# ---- VERIFY THE 8 CLIMATE FEATURES ARE PRESENT ----
print("\nClimate features in final dataset:")
climate_features = ['CDD26', 'mean_temp', 'mean_temp_summer', 'max_temp',
                    'RH_mean', 'HI_mean', 'ts_anom', 'ts_anom_summer']
for c in climate_features:
    status = "[OK]" if c in ml_dataset.columns else "[MISSING]"
    print(f"  {status} {c}")

# ---- MISSING VALUE PROFILE OF REMAINING COLUMNS ----
print("\n" + "="*60)
print("MISSING VALUE PROFILE (remaining columns):")
print("="*60)
miss = (ml_dataset.isnull().mean() * 100).sort_values(ascending=False)

# Buckets to understand the data quality
buckets = {
    "  0% missing (fully complete)":    (miss == 0).sum(),
    "  0-10% missing (very clean)":     ((miss > 0) & (miss <= 10)).sum(),
    " 10-30% missing (usable)":         ((miss > 10) & (miss <= 30)).sum(),
    " 30-50% missing (sparse)":         ((miss > 30) & (miss <= 50)).sum(),
    " 50-90% missing (very sparse)":    ((miss > 50) & (miss <= 90)).sum(),
    " 90-99% missing (near-empty)":     ((miss > 90) & (miss < 100)).sum(),
}
for bucket, count in buckets.items():
    print(f"  {bucket:<35} : {count:>3} columns")

print(f"\n[OK] {len(ml_dataset.columns)} columns remain in the labeled dataset.")
print("\nDone.")

Dropping 85 columns from the labeled dataset...

Shape BEFORE drop: 19,063 rows  ×  226 columns
Shape AFTER drop:  19,063 rows  ×  141 columns
Columns removed:   85

[OK] Target variable CLIMATIZZAZIONE_ESTIVA is preserved.

Climate features in final dataset:
  [OK] CDD26
  [OK] mean_temp
  [OK] mean_temp_summer
  [OK] max_temp
  [OK] RH_mean
  [OK] HI_mean
  [OK] ts_anom
  [OK] ts_anom_summer

MISSING VALUE PROFILE (remaining columns):
    0% missing (fully complete)       :  69 columns
    0-10% missing (very clean)        :  26 columns
   10-30% missing (usable)            :  14 columns
   30-50% missing (sparse)            :   0 columns
   50-90% missing (very sparse)       :   9 columns
   90-99% missing (near-empty)        :  23 columns

[OK] 141 columns remain in the labeled dataset.

Done.


In [13]:
# =============================================================
# STEP 3 — CELL 10
# Save the final clean Stage 1 ML dataset and full traceability log
# =============================================================

from datetime import datetime

print("Saving final ML dataset and traceability log...\n")

# ---- 1. SAVE THE FINAL ML DATASET ----
output_path = FINAL_DIR / "stage1_ml_dataset_v1.csv"
print(f"Saving ML dataset to: {output_path.name}")
print("  (This may take ~30 seconds for ~25 MB...)")

ml_dataset.to_csv(output_path, index=False)
size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"  [OK] Saved ({size_mb:.2f} MB)")

# ---- 2. CREATE A TRACEABILITY LOG ----
# Documents every dropped column and its category — essential for thesis audit trail
log_rows = []
for col in sorted(columns_to_drop):
    if col in empty_cols:
        category = "1_EMPTY"
        reason = "100% missing values"
    elif col in leakage_cols:
        category = "3_LEAKAGE"
        reason = "directly reveals cooling target"
    elif col in identifier_cols:
        category = "4_IDENTIFIER"
        reason = "ID / non-feature column"
    elif col in constant_cols:
        category = "2_CONSTANT"
        reason = "single unique value (zero info)"
    else:
        category = "OTHER"
        reason = "review needed"
    log_rows.append({'column': col, 'category': category, 'reason': reason})

log_df = pd.DataFrame(log_rows).sort_values(['category', 'column']).reset_index(drop=True)
log_path = LOGS_DIR / "step3_dropped_columns_log.csv"
log_df.to_csv(log_path, index=False)
print(f"\nTraceability log saved to: 05_logs/{log_path.name}")
print(f"  ({len(log_df)} dropped columns documented)")

# ---- 3. SAVE A README WITH FULL CONTEXT ----
readme_path = LOGS_DIR / "step3_README.txt"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("STEP 3 — STAGE 1 ML DATASET CONSTRUCTION\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Date generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write("PURPOSE:\n")
    f.write("  Build a clean, leakage-free dataset for Stage 1 of the ML pipeline\n")
    f.write("  (binary classification: predict cooling presence).\n\n")
    f.write("INPUT:\n")
    f.write("  03_intermediate/buildings_with_epc_and_climate_v1.csv\n")
    f.write("  (53,041 buildings × 226 columns)\n\n")
    f.write("OUTPUT:\n")
    f.write("  04_final_ml_dataset/stage1_ml_dataset_v1.csv\n")
    f.write(f"  (19,063 buildings × 141 columns)\n\n")
    f.write("TARGET VARIABLE:\n")
    f.write("  CLIMATIZZAZIONE_ESTIVA (boolean): True = building has cooling system\n\n")
    f.write("CLASS BALANCE:\n")
    f.write("  Cooling YES: 13,787 (72.3%)\n")
    f.write("  Cooling NO :  5,276 (27.7%)\n")
    f.write("  Imbalance ratio = 2.61 : 1 (moderate, manageable)\n\n")
    f.write("PROCESSING APPLIED:\n")
    f.write("  1. Filtered to 19,063 buildings with EPC labels\n")
    f.write("  2. Dropped 85 columns:\n")
    f.write(f"     - {len(empty_cols)} empty columns (100% missing)\n")
    f.write(f"     - {len(constant_cols)-len(removed_climate)} constant columns\n")
    f.write(f"     - {len(leakage_cols)-len(saved_design)} leakage columns (cooling-related)\n")
    f.write(f"     - {len(identifier_cols)} identifier / non-feature columns\n\n")
    f.write("FEATURES PROTECTED (kept despite appearing 'constant'):\n")
    f.write("  Climate features (essential for Stage 2 future scenarios):\n")
    for c in climate_features:
        f.write(f"    - {c}\n")
    f.write("  Building design feature:\n")
    for c in saved_design:
        f.write(f"    - {c}\n")
    f.write("\nLEAKAGE EVIDENCE:\n")
    f.write("  Verified empirically using epc_clean_final.csv (342,684 records):\n")
    f.write("  - SUPERF_UTILE_RAFFRESCATA = 0 for ALL non-cooled buildings (perfect leakage)\n")
    f.write("  - VOLUME_LORDO_RAFFRESCATO = 0 for ALL non-cooled buildings (perfect leakage)\n")
    f.write("  - CE_* columns describe the cooling system itself (leakage)\n\n")
    f.write("NEXT STEPS:\n")
    f.write("  Step 4: Feature engineering and selection refinement\n")
    f.write("  Step 5: Train Stage 1 classifier (Logistic Regression / RF / XGBoost)\n")
    f.write("  Step 6: Evaluate model and apply SHAP for interpretability\n")
print(f"\nDocumentation saved to: 05_logs/{readme_path.name}")

# ---- 4. FINAL SUMMARY ----
print("\n" + "="*65)
print("STEP 3 — COMPLETE")
print("="*65)
print(f"  Input:     53,041 × 226 (full merged dataset)")
print(f"  Output:    19,063 × 141 (Stage 1 ML dataset)")
print(f"  Target:    CLIMATIZZAZIONE_ESTIVA (binary)")
print(f"  Files written:")
print(f"    1. 04_final_ml_dataset/stage1_ml_dataset_v1.csv")
print(f"    2. 05_logs/step3_dropped_columns_log.csv")
print(f"    3. 05_logs/step3_README.txt")
print("\nReady for Step 4 (Feature engineering & model preparation).")
print("\nDone.")

Saving final ML dataset and traceability log...

Saving ML dataset to: stage1_ml_dataset_v1.csv
  (This may take ~30 seconds for ~25 MB...)
  [OK] Saved (17.87 MB)

Traceability log saved to: 05_logs/step3_dropped_columns_log.csv
  (85 dropped columns documented)

Documentation saved to: 05_logs/step3_README.txt

STEP 3 — COMPLETE
  Input:     53,041 × 226 (full merged dataset)
  Output:    19,063 × 141 (Stage 1 ML dataset)
  Target:    CLIMATIZZAZIONE_ESTIVA (binary)
  Files written:
    1. 04_final_ml_dataset/stage1_ml_dataset_v1.csv
    2. 05_logs/step3_dropped_columns_log.csv
    3. 05_logs/step3_README.txt

Ready for Step 4 (Feature engineering & model preparation).

Done.


In [1]:
# =============================================================
# STEP 4 — CELL 11
# Define candidate feature list and inspect each one
# =============================================================

print("Inspecting candidate features for Stage 1 ML...\n")

# ---- DEFINE CANDIDATE FEATURES (CONSERVATIVE APPROACH) ----
# Organized in 4 logical groups for clarity and traceability.
candidate_features = {
    # GROUP A — DBT geometry attributes (always 0% missing in DBT)
    'A_DBT_geometry': [
        'EDIFC_TY',      # building type (residential / non-residential / mixed / etc.)
        'EDIFC_USO',     # specific use code
        'EDIFC_STAT',    # building status (in use / under construction / abandoned)
        'EDIFC_CASS',    # construction class (e.g., masonry, reinforced concrete)
    ],

    # GROUP B — EPC core building information (safe predictors)
    'B_EPC_building_info': [
        'ANNO_COSTRUZIONE',              # construction year
        'SUPERF_UTILE_RISCALDATA',       # heated useful surface (m²) — NOT cooled, safe
        'VOLUME_LORDO_RISCALDATO',       # heated gross volume (m³)
        'RAPPORTO_SV',                   # surface-to-volume ratio
        'SUPERFICIE_DISPERDENTE',        # dispersing envelope surface (m²)
        'NUMERO_UNITA_IMMOBILIARI',      # number of real-estate units
        'CLASSIFICAZIONE_DPR',           # DPR 412/93 building category
        'INTERO_EDIFICIO',               # certificate covers entire building (yes/no)
        'NUOVA_COSTRUZIONE',             # new construction (yes/no)
        'RISTRUTTURAZIONE_IMPORTANTE',   # major renovation (yes/no)
        'RIQUALIFICAZIONE_ENERGETICA',   # energy retrofit (yes/no)
        'PASSAGGIO_PROPRIETA',           # property transfer (yes/no)
        'LOCAZIONE',                     # rental contract (yes/no)
    ],

    # GROUP C — Building design parameter (verified safe in Step 3)
    'C_design': [
        'A_SOL_EST_A_SUP_UTILE',         # summer solar gain ratio
    ],

    # GROUP D — Climate features (essential for Stage 2 future scenarios)
    'D_climate': [
        'CDD26', 'mean_temp', 'mean_temp_summer', 'max_temp',
        'RH_mean', 'HI_mean', 'ts_anom', 'ts_anom_summer',
    ],
}

# Flatten for easy iteration
all_candidates = []
for group, cols in candidate_features.items():
    all_candidates.extend(cols)

print(f"Total candidate features: {len(all_candidates)}")
for group, cols in candidate_features.items():
    print(f"  {group}: {len(cols)} features")

# ---- INSPECT EACH CANDIDATE ----
print("\n" + "="*90)
print(f"{'FEATURE':<32} {'DTYPE':<10} {'MISS%':>7} {'UNIQUE':>7}   SAMPLE")
print("="*90)

flagged = []
for group, cols in candidate_features.items():
    print(f"\n--- {group} ---")
    for col in cols:
        if col not in ml_dataset.columns:
            print(f"  {col:<32} *** NOT IN DATASET ***")
            flagged.append((col, 'missing'))
            continue

        s = ml_dataset[col]
        dtype  = str(s.dtype)[:9]
        miss   = s.isnull().mean() * 100
        nuniq  = s.nunique(dropna=True)

        # Sample values (top 2 most frequent for readability)
        if nuniq <= 6:
            sample = str(s.dropna().value_counts().head(3).to_dict())[:42]
        elif s.dtype.kind in 'fi':
            sample = f"min={s.min():.1f}, max={s.max():.1f}, median={s.median():.1f}"
        else:
            sample = str(s.dropna().value_counts().head(2).index.tolist())[:42]

        flag = ""
        if miss > 50:
            flag = "  [HIGH MISS]"
            flagged.append((col, f'{miss:.0f}% missing'))
        elif nuniq <= 1:
            flag = "  [CONSTANT]"
            flagged.append((col, 'constant'))

        print(f"  {col:<32} {dtype:<10} {miss:>6.1f}% {nuniq:>7}   {sample}{flag}")

# ---- SUMMARY OF ANY FLAGS ----
print("\n" + "="*90)
if flagged:
    print(f"FLAGGED features needing decision ({len(flagged)}):")
    for col, reason in flagged:
        print(f"  - {col}  ({reason})")
else:
    print("[OK] No flagged features. All candidates are usable.")
print("="*90)
print("\nDone.")

Inspecting candidate features for Stage 1 ML...

Total candidate features: 26
  A_DBT_geometry: 4 features
  B_EPC_building_info: 13 features
  C_design: 1 features
  D_climate: 8 features

FEATURE                          DTYPE        MISS%  UNIQUE   SAMPLE

--- A_DBT_geometry ---


NameError: name 'ml_dataset' is not defined

In [2]:
# =============================================================
# CELL 10b — Reload checkpoint after kernel restart
# Reloads the Stage 1 ML dataset and re-defines paths/imports
# Run this cell first if you reopen the notebook
# =============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ---- RE-DEFINE PROJECT PATHS ----
PROJECT_ROOT     = Path(r"C:\Users\n.mohammadi\Desktop\NimaMohammadi\02.Nima Mohammadi - Thesis\ML Dataset")
INTERMEDIATE_DIR = PROJECT_ROOT / "03_intermediate"
FINAL_DIR        = PROJECT_ROOT / "04_final_ml_dataset"
LOGS_DIR         = PROJECT_ROOT / "05_logs"

# ---- RELOAD STAGE 1 ML DATASET ----
input_path = FINAL_DIR / "stage1_ml_dataset_v1.csv"
print(f"Reloading: {input_path.name}")
ml_dataset = pd.read_csv(input_path, low_memory=False)

print(f"  [OK] Shape: {ml_dataset.shape[0]:,} rows × {ml_dataset.shape[1]} columns\n")

# ---- VERIFY KEY COLUMNS ARE PRESENT ----
critical_cols = ['CLIMATIZZAZIONE_ESTIVA', 'EDIFC_TY', 'EDIFC_USO',
                 'ANNO_COSTRUZIONE', 'CDD26', 'mean_temp_summer']
print("Critical columns check:")
for c in critical_cols:
    status = "[OK]" if c in ml_dataset.columns else "[MISSING]"
    print(f"  {status} {c}")

# ---- VERIFY TARGET DISTRIBUTION ----
print(f"\nTarget distribution:")
counts = ml_dataset['CLIMATIZZAZIONE_ESTIVA'].value_counts()
for value, count in counts.items():
    pct = count / len(ml_dataset) * 100
    label = "Cooling YES" if value is True else "Cooling NO"
    print(f"  {label}: {count:,} ({pct:.1f}%)")

print("\nDone. Ready to continue with Cell 11.")

Reloading: stage1_ml_dataset_v1.csv
  [OK] Shape: 19,063 rows × 141 columns

Critical columns check:
  [OK] CLIMATIZZAZIONE_ESTIVA
  [OK] EDIFC_TY
  [OK] EDIFC_USO
  [OK] ANNO_COSTRUZIONE
  [OK] CDD26
  [OK] mean_temp_summer

Target distribution:
  Cooling YES: 13,787 (72.3%)
  Cooling NO: 5,276 (27.7%)

Done. Ready to continue with Cell 11.


In [3]:
# =============================================================
# STEP 4 — CELL 11
# Define candidate feature list and inspect each one
# =============================================================

print("Inspecting candidate features for Stage 1 ML...\n")

# ---- DEFINE CANDIDATE FEATURES (CONSERVATIVE APPROACH) ----
# Organized in 4 logical groups for clarity and traceability.
candidate_features = {
    # GROUP A — DBT geometry attributes (always 0% missing in DBT)
    'A_DBT_geometry': [
        'EDIFC_TY',      # building type (residential / non-residential / mixed / etc.)
        'EDIFC_USO',     # specific use code
        'EDIFC_STAT',    # building status (in use / under construction / abandoned)
        'EDIFC_CASS',    # construction class (e.g., masonry, reinforced concrete)
    ],

    # GROUP B — EPC core building information (safe predictors)
    'B_EPC_building_info': [
        'ANNO_COSTRUZIONE',              # construction year
        'SUPERF_UTILE_RISCALDATA',       # heated useful surface (m²) — NOT cooled, safe
        'VOLUME_LORDO_RISCALDATO',       # heated gross volume (m³)
        'RAPPORTO_SV',                   # surface-to-volume ratio
        'SUPERFICIE_DISPERDENTE',        # dispersing envelope surface (m²)
        'NUMERO_UNITA_IMMOBILIARI',      # number of real-estate units
        'CLASSIFICAZIONE_DPR',           # DPR 412/93 building category
        'INTERO_EDIFICIO',               # certificate covers entire building (yes/no)
        'NUOVA_COSTRUZIONE',             # new construction (yes/no)
        'RISTRUTTURAZIONE_IMPORTANTE',   # major renovation (yes/no)
        'RIQUALIFICAZIONE_ENERGETICA',   # energy retrofit (yes/no)
        'PASSAGGIO_PROPRIETA',           # property transfer (yes/no)
        'LOCAZIONE',                     # rental contract (yes/no)
    ],

    # GROUP C — Building design parameter (verified safe in Step 3)
    'C_design': [
        'A_SOL_EST_A_SUP_UTILE',         # summer solar gain ratio
    ],

    # GROUP D — Climate features (essential for Stage 2 future scenarios)
    'D_climate': [
        'CDD26', 'mean_temp', 'mean_temp_summer', 'max_temp',
        'RH_mean', 'HI_mean', 'ts_anom', 'ts_anom_summer',
    ],
}

# Flatten for easy iteration
all_candidates = []
for group, cols in candidate_features.items():
    all_candidates.extend(cols)

print(f"Total candidate features: {len(all_candidates)}")
for group, cols in candidate_features.items():
    print(f"  {group}: {len(cols)} features")

# ---- INSPECT EACH CANDIDATE ----
print("\n" + "="*90)
print(f"{'FEATURE':<32} {'DTYPE':<10} {'MISS%':>7} {'UNIQUE':>7}   SAMPLE")
print("="*90)

flagged = []
for group, cols in candidate_features.items():
    print(f"\n--- {group} ---")
    for col in cols:
        if col not in ml_dataset.columns:
            print(f"  {col:<32} *** NOT IN DATASET ***")
            flagged.append((col, 'missing'))
            continue

        s = ml_dataset[col]
        dtype  = str(s.dtype)[:9]
        miss   = s.isnull().mean() * 100
        nuniq  = s.nunique(dropna=True)

        # Sample values (top 2 most frequent for readability)
        if nuniq <= 6:
            sample = str(s.dropna().value_counts().head(3).to_dict())[:42]
        elif s.dtype.kind in 'fi':
            sample = f"min={s.min():.1f}, max={s.max():.1f}, median={s.median():.1f}"
        else:
            sample = str(s.dropna().value_counts().head(2).index.tolist())[:42]

        flag = ""
        if miss > 50:
            flag = "  [HIGH MISS]"
            flagged.append((col, f'{miss:.0f}% missing'))
        elif nuniq <= 1:
            flag = "  [CONSTANT]"
            flagged.append((col, 'constant'))

        print(f"  {col:<32} {dtype:<10} {miss:>6.1f}% {nuniq:>7}   {sample}{flag}")

# ---- SUMMARY OF ANY FLAGS ----
print("\n" + "="*90)
if flagged:
    print(f"FLAGGED features needing decision ({len(flagged)}):")
    for col, reason in flagged:
        print(f"  - {col}  ({reason})")
else:
    print("[OK] No flagged features. All candidates are usable.")
print("="*90)
print("\nDone.")

Inspecting candidate features for Stage 1 ML...

Total candidate features: 26
  A_DBT_geometry: 4 features
  B_EPC_building_info: 13 features
  C_design: 1 features
  D_climate: 8 features

FEATURE                          DTYPE        MISS%  UNIQUE   SAMPLE

--- A_DBT_geometry ---
  EDIFC_TY                         int64         0.0%       6   {101: 17673, 108: 1286, 111: 64}
  EDIFC_USO                        int64         0.0%      21   min=201.0, max=2080206.0, median=201.0
  EDIFC_STAT                       int64         0.0%       3   {403: 18943, 401: 90, 402: 30}
  EDIFC_CASS                       int64         0.0%   19063   min=1.0, max=53049.0, median=25162.0

--- B_EPC_building_info ---
  ANNO_COSTRUZIONE                 str           0.0%     146   ['1961-1976', '1946-1960']
  SUPERF_UTILE_RISCALDATA          float64       0.0%    9542   min=3.4, max=84477.9, median=67.6
  VOLUME_LORDO_RISCALDATO          float64       0.0%   15263   min=20.7, max=3460005.0, median=264.9
 

In [4]:
# =============================================================
# STEP 4 — CELL 12
# Apply feature engineering transformations (with full traceability)
# =============================================================

print("Applying feature engineering transformations...\n")

# ---- WORK ON A COPY (preserve ml_dataset for safety) ----
df = ml_dataset.copy()

# =============================================================
# TRANSFORMATION 1 — Remove EDIFC_CASS (it is an identifier)
# =============================================================
print("[1] Removing EDIFC_CASS from feature list")
print("    Justification: 19,063 unique values across 19,063 rows = unique ID, not a feature.")
print("    Action: dropped from candidate list (still present in dataset for traceability).\n")

# =============================================================
# TRANSFORMATION 2 — Convert ANNO_COSTRUZIONE to numeric midpoints
# =============================================================
print("[2] Converting ANNO_COSTRUZIONE from text ranges to numeric midpoints")

def parse_anno_costruzione(value):
    """Convert string ranges like '1961-1976' or '<1900' or '1900-1918' to numeric midpoint."""
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    # Handle '<1900' style values
    if s.startswith('<'):
        try:
            return float(s.replace('<', '')) - 5  # e.g., <1900 → 1895
        except ValueError:
            return np.nan
    # Handle '>2020' style values
    if s.startswith('>'):
        try:
            return float(s.replace('>', '')) + 5  # e.g., >2020 → 2025
        except ValueError:
            return np.nan
    # Handle 'YYYY-YYYY' ranges
    if '-' in s:
        try:
            parts = s.split('-')
            y1 = float(parts[0].strip())
            y2 = float(parts[1].strip())
            return (y1 + y2) / 2
        except (ValueError, IndexError):
            return np.nan
    # Handle single year
    try:
        return float(s)
    except ValueError:
        return np.nan

df['construction_year'] = df['ANNO_COSTRUZIONE'].apply(parse_anno_costruzione)
df['building_age'] = 2024 - df['construction_year']

n_parsed = df['construction_year'].notna().sum()
n_failed = df['construction_year'].isna().sum()
print(f"    Parsed successfully: {n_parsed:,} ({n_parsed/len(df)*100:.1f}%)")
print(f"    Failed to parse:     {n_failed:,} ({n_failed/len(df)*100:.1f}%)")
print(f"    Year range:          {df['construction_year'].min():.0f} – {df['construction_year'].max():.0f}")
print(f"    Median year:         {df['construction_year'].median():.0f}")
print(f"    Median building age: {df['building_age'].median():.0f} years")

# Show sample mapping for verification
print("\n    Sample of conversions (top 5 most frequent ranges):")
sample_rows = df[['ANNO_COSTRUZIONE', 'construction_year', 'building_age']].dropna()
sample_rows = sample_rows.drop_duplicates(subset='ANNO_COSTRUZIONE').head(5)
for _, r in sample_rows.iterrows():
    print(f"      {r['ANNO_COSTRUZIONE']:<15} → year={r['construction_year']:.0f}, age={r['building_age']:.0f}")

# =============================================================
# TRANSFORMATION 3 — Create EDIFC_USO_main (first-digit grouping)
# =============================================================
print("\n[3] Creating EDIFC_USO_main (simplified categorical from EDIFC_USO)")

def first_digit_group(code):
    """Italian cadastre codes are hierarchical; first digit gives broad category."""
    if pd.isna(code):
        return np.nan
    return str(int(code))[0]

df['EDIFC_USO_main'] = df['EDIFC_USO'].apply(first_digit_group)
print(f"    Distribution of EDIFC_USO_main:")
print(df['EDIFC_USO_main'].value_counts().to_string())

# =============================================================
# UPDATED CANDIDATE FEATURE LIST
# =============================================================
print("\n" + "="*70)
print("UPDATED CANDIDATE FEATURE LIST (Stage 1 ML)")
print("="*70)

final_features = {
    'A_DBT_geometry': [
        'EDIFC_TY',          # building type
        'EDIFC_USO',         # full hierarchical use code (categorical)
        'EDIFC_USO_main',    # simplified first-digit group (derived)
        'EDIFC_STAT',        # building status
    ],
    'B_EPC_building_info': [
        'construction_year',           # numeric (derived from ANNO_COSTRUZIONE)
        'building_age',                # numeric (derived: 2024 - construction_year)
        'SUPERF_UTILE_RISCALDATA',     # heated useful surface
        'VOLUME_LORDO_RISCALDATO',     # heated gross volume
        'RAPPORTO_SV',                 # surface-to-volume ratio
        'SUPERFICIE_DISPERDENTE',      # dispersing envelope surface
        'NUMERO_UNITA_IMMOBILIARI',    # number of real-estate units
        'CLASSIFICAZIONE_DPR',         # DPR 412/93 building category
        'INTERO_EDIFICIO',
        'NUOVA_COSTRUZIONE',
        'RISTRUTTURAZIONE_IMPORTANTE',
        'RIQUALIFICAZIONE_ENERGETICA',
        'PASSAGGIO_PROPRIETA',
        'LOCAZIONE',
    ],
    'C_design': [
        'A_SOL_EST_A_SUP_UTILE',
    ],
    'D_climate': [
        'CDD26', 'mean_temp', 'mean_temp_summer', 'max_temp',
        'RH_mean', 'HI_mean', 'ts_anom', 'ts_anom_summer',
    ],
}

n_total = 0
for group, cols in final_features.items():
    print(f"\n{group}: {len(cols)} features")
    for c in cols:
        present = "[OK]" if c in df.columns else "[MISSING]"
        print(f"  {present} {c}")
    n_total += len(cols)

print(f"\nTotal candidate features: {n_total}")
print(f"(was 26, removed 1 = EDIFC_CASS, added 3 derived: construction_year, building_age, EDIFC_USO_main)")

print("\nDone. Working dataset 'df' is ready for Cell 13 (encoding & assembly).")

Applying feature engineering transformations...

[1] Removing EDIFC_CASS from feature list
    Justification: 19,063 unique values across 19,063 rows = unique ID, not a feature.
    Action: dropped from candidate list (still present in dataset for traceability).

[2] Converting ANNO_COSTRUZIONE from text ranges to numeric midpoints
    Parsed successfully: 16,119 (84.6%)
    Failed to parse:     2,944 (15.4%)
    Year range:          1500 – 2025
    Median year:         1960
    Median building age: 64 years

    Sample of conversions (top 5 most frequent ranges):
      1961-1976       → year=1968, age=56
      2004            → year=2004, age=20
      1946-1960       → year=1953, age=71
      1977-1992       → year=1984, age=40
      2005            → year=2005, age=19

[3] Creating EDIFC_USO_main (simplified categorical from EDIFC_USO)
    Distribution of EDIFC_USO_main:
EDIFC_USO_main
2    19063

UPDATED CANDIDATE FEATURE LIST (Stage 1 ML)

A_DBT_geometry: 4 features
  [OK] EDIFC_TY

In [5]:
# =============================================================
# STEP 4 — CELL 12b
# Diagnose and fix issues from Cell 12
# =============================================================

print("Diagnostic check on the issues raised by Cell 12...\n")

# =============================================================
# ISSUE 1 — EDIFC_USO_main is a constant column
# =============================================================
print("[1] EDIFC_USO_main analysis:")
print(f"    Unique values: {df['EDIFC_USO_main'].nunique()}")
print(f"    -> Removing from feature list. Reverting to original EDIFC_USO only.\n")

# =============================================================
# ISSUE 2 — Investigate failed ANNO_COSTRUZIONE values
# =============================================================
print("[2] Investigating the 2,944 failed ANNO_COSTRUZIONE values:")
failed_mask = df['construction_year'].isna()
failed_values = df.loc[failed_mask, 'ANNO_COSTRUZIONE']

print(f"    Total failed: {len(failed_values):,}")
print(f"    Unique failed values: {failed_values.nunique()}\n")

print("    Top 15 most common failed values:")
print(failed_values.value_counts().head(15).to_string())

# Show a few sample failed values to understand patterns
print("\n    Length distribution of failed values:")
length_dist = failed_values.str.len().value_counts().sort_index().head(10)
for length, count in length_dist.items():
    sample = failed_values[failed_values.str.len() == length].head(3).tolist()
    print(f"      length={length}: {count:>5} occurrences | examples: {sample}")

Diagnostic check on the issues raised by Cell 12...

[1] EDIFC_USO_main analysis:
    Unique values: 1
    -> Removing from feature list. Reverting to original EDIFC_USO only.

[2] Investigating the 2,944 failed ANNO_COSTRUZIONE values:
    Total failed: 2,944
    Unique failed values: 2

    Top 15 most common failed values:
ANNO_COSTRUZIONE
Prima del 1930    2532
Dopo il 2006       412

    Length distribution of failed values:
      length=12:   412 occurrences | examples: ['Dopo il 2006', 'Dopo il 2006', 'Dopo il 2006']
      length=14:  2532 occurrences | examples: ['Prima del 1930', 'Prima del 1930', 'Prima del 1930']


In [6]:
# =============================================================
# STEP 4 — CELL 12c
# Fix ANNO_COSTRUZIONE parsing and clean up feature list
# =============================================================

print("Applying fixes from Cell 12b diagnostics...\n")

# =============================================================
# FIX 1 — Improved ANNO_COSTRUZIONE parser
# =============================================================
print("[1] Re-parsing ANNO_COSTRUZIONE with extended logic")

def parse_anno_costruzione_v2(value):
    """Robust parser that handles all observed CENED formats."""
    if pd.isna(value):
        return np.nan
    s = str(value).strip()

    # Italian open-ended phrasing
    if s.lower().startswith('prima del'):
        try:
            year_limit = int(''.join(c for c in s if c.isdigit()))
            return year_limit - 10  # e.g., "Prima del 1930" -> 1920
        except ValueError:
            return np.nan

    if s.lower().startswith('dopo il'):
        try:
            year_limit = int(''.join(c for c in s if c.isdigit()))
            return year_limit + 9   # e.g., "Dopo il 2006" -> 2015
        except ValueError:
            return np.nan

    # Mathematical open-ended phrasing
    if s.startswith('<'):
        try:
            return float(s.replace('<', '').strip()) - 5
        except ValueError:
            return np.nan
    if s.startswith('>'):
        try:
            return float(s.replace('>', '').strip()) + 5
        except ValueError:
            return np.nan

    # Range "YYYY-YYYY"
    if '-' in s:
        try:
            parts = s.split('-')
            y1 = float(parts[0].strip())
            y2 = float(parts[1].strip())
            return (y1 + y2) / 2
        except (ValueError, IndexError):
            return np.nan

    # Single year
    try:
        return float(s)
    except ValueError:
        return np.nan

# Apply the new parser
df['construction_year'] = df['ANNO_COSTRUZIONE'].apply(parse_anno_costruzione_v2)
df['building_age']      = 2024 - df['construction_year']

# ---- VERIFY ----
n_parsed = df['construction_year'].notna().sum()
n_failed = df['construction_year'].isna().sum()
print(f"    Parsed successfully: {n_parsed:,} ({n_parsed/len(df)*100:.1f}%)")
print(f"    Failed to parse:     {n_failed:,} ({n_failed/len(df)*100:.1f}%)")

if n_failed == 0:
    print(f"    [OK] All 19,063 buildings now have a numeric construction year.\n")
else:
    print(f"    [WARNING] Still {n_failed} unparsed — check before proceeding.\n")

# Show the distribution after the fix
print("    Construction year distribution (deciles):")
for q in [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]:
    val = df['construction_year'].quantile(q)
    print(f"      {int(q*100):>3}th percentile: {val:.0f}")

# Verify the two special cases were correctly mapped
print("\n    Verification of special cases:")
for special in ['Prima del 1930', 'Dopo il 2006']:
    sample = df.loc[df['ANNO_COSTRUZIONE'] == special,
                    ['ANNO_COSTRUZIONE', 'construction_year', 'building_age']].head(1)
    if not sample.empty:
        r = sample.iloc[0]
        print(f"      '{r['ANNO_COSTRUZIONE']}' -> year={r['construction_year']:.0f}, age={r['building_age']:.0f}")

# =============================================================
# FIX 2 — Remove EDIFC_USO_main from feature list
# =============================================================
print("\n[2] Removing EDIFC_USO_main from feature list (constant column)")

# =============================================================
# FINAL CLEAN FEATURE LIST
# =============================================================
final_features = {
    'A_DBT_geometry': [
        'EDIFC_TY',
        'EDIFC_USO',
        'EDIFC_STAT',
    ],
    'B_EPC_building_info': [
        'construction_year',
        'building_age',
        'SUPERF_UTILE_RISCALDATA',
        'VOLUME_LORDO_RISCALDATO',
        'RAPPORTO_SV',
        'SUPERFICIE_DISPERDENTE',
        'NUMERO_UNITA_IMMOBILIARI',
        'CLASSIFICAZIONE_DPR',
        'INTERO_EDIFICIO',
        'NUOVA_COSTRUZIONE',
        'RISTRUTTURAZIONE_IMPORTANTE',
        'RIQUALIFICAZIONE_ENERGETICA',
        'PASSAGGIO_PROPRIETA',
        'LOCAZIONE',
    ],
    'C_design': [
        'A_SOL_EST_A_SUP_UTILE',
    ],
    'D_climate': [
        'CDD26', 'mean_temp', 'mean_temp_summer', 'max_temp',
        'RH_mean', 'HI_mean', 'ts_anom', 'ts_anom_summer',
    ],
}

print("\n" + "="*65)
print("FINAL CANDIDATE FEATURE LIST (after fixes)")
print("="*65)

n_total = 0
for group, cols in final_features.items():
    print(f"\n{group}: {len(cols)} features")
    for c in cols:
        present_status = "[OK]" if c in df.columns else "[MISSING]"
        miss_pct = df[c].isnull().mean() * 100 if c in df.columns else 100
        miss_flag = f"  miss={miss_pct:.1f}%" if miss_pct > 0 else ""
        print(f"  {present_status} {c}{miss_flag}")
    n_total += len(cols)

print(f"\nTotal candidate features: {n_total}")
print("\nDone. Working dataset 'df' is fully clean and ready for Cell 13.")

Applying fixes from Cell 12b diagnostics...

[1] Re-parsing ANNO_COSTRUZIONE with extended logic
    Parsed successfully: 19,063 (100.0%)
    Failed to parse:     0 (0.0%)
    [OK] All 19,063 buildings now have a numeric construction year.

    Construction year distribution (deciles):
        0th percentile: 1500
       10th percentile: 1920
       25th percentile: 1938
       50th percentile: 1953
       75th percentile: 1968
       90th percentile: 1992
      100th percentile: 2025

    Verification of special cases:
      'Prima del 1930' -> year=1920, age=104
      'Dopo il 2006' -> year=2015, age=9

[2] Removing EDIFC_USO_main from feature list (constant column)

FINAL CANDIDATE FEATURE LIST (after fixes)

A_DBT_geometry: 3 features
  [OK] EDIFC_TY
  [OK] EDIFC_USO
  [OK] EDIFC_STAT

B_EPC_building_info: 14 features
  [OK] construction_year
  [OK] building_age
  [OK] SUPERF_UTILE_RISCALDATA
  [OK] VOLUME_LORDO_RISCALDATO
  [OK] RAPPORTO_SV
  [OK] SUPERFICIE_DISPERDENTE
  [OK] NUM

In [7]:
# =============================================================
# STEP 4 — CELL 13
# Assemble working subset and classify feature types
# =============================================================

print("Assembling the final working subset...\n")

# ---- BUILD THE FLAT LIST OF FEATURES ----
all_features = []
for cols in final_features.values():
    all_features.extend(cols)

target_col = 'CLIMATIZZAZIONE_ESTIVA'

# ---- EXTRACT WORKING SUBSET ----
working_df = df[all_features + [target_col]].copy()

print(f"Working subset shape: {working_df.shape[0]:,} rows × {working_df.shape[1]} columns")
print(f"  ({len(all_features)} features + 1 target)\n")

# ---- CLASSIFY FEATURE TYPES ----
numeric_features = []
boolean_features = []
categorical_features = []

for f in all_features:
    s = working_df[f]
    if s.dtype == bool:
        boolean_features.append(f)
    elif pd.api.types.is_numeric_dtype(s):
        # Numeric, but check if it looks categorical (low unique count + integer codes)
        nuniq = s.nunique()
        if s.dtype.kind == 'i' and nuniq <= 25 and f in ['EDIFC_TY', 'EDIFC_USO', 'EDIFC_STAT']:
            categorical_features.append(f)
        else:
            numeric_features.append(f)
    else:
        categorical_features.append(f)

print("="*70)
print("FEATURE TYPE CLASSIFICATION")
print("="*70)

print(f"\nNUMERIC features ({len(numeric_features)}):")
for f in numeric_features:
    s = working_df[f]
    print(f"  {f:<32}  range=[{s.min():.2f}, {s.max():.2f}]  median={s.median():.2f}")

print(f"\nBOOLEAN features ({len(boolean_features)}):")
for f in boolean_features:
    vc = working_df[f].value_counts()
    true_pct  = vc.get(True, 0)  / len(working_df) * 100
    print(f"  {f:<32}  TRUE = {true_pct:5.1f}%")

print(f"\nCATEGORICAL features ({len(categorical_features)}):")
for f in categorical_features:
    nuniq = working_df[f].nunique()
    top3  = working_df[f].value_counts().head(3).to_dict()
    print(f"  {f:<32}  unique={nuniq:>3}  top3={top3}")

# ---- TARGET CHECK ----
print("\n" + "="*70)
print("TARGET VARIABLE")
print("="*70)
target_dist = working_df[target_col].value_counts()
print(f"  Cooling YES (True):  {target_dist[True]:>6,}  ({target_dist[True]/len(working_df)*100:.1f}%)")
print(f"  Cooling NO  (False): {target_dist[False]:>6,}  ({target_dist[False]/len(working_df)*100:.1f}%)")

# ---- STORE LISTS FOR CELL 14 ----
print("\n" + "="*70)
print("FEATURE TYPE COUNTS:")
print(f"  Numeric:     {len(numeric_features):>3}")
print(f"  Boolean:     {len(boolean_features):>3}")
print(f"  Categorical: {len(categorical_features):>3}")
print(f"  TOTAL:       {len(numeric_features) + len(boolean_features) + len(categorical_features):>3}")
print("="*70)

print("\nDone. Ready for Cell 14 (encoding + final assembly).")

Assembling the final working subset...

Working subset shape: 19,063 rows × 27 columns
  (26 features + 1 target)

FEATURE TYPE CLASSIFICATION

NUMERIC features (16):
  construction_year                 range=[1500.00, 2025.00]  median=1953.00
  building_age                      range=[-1.00, 524.00]  median=71.00
  SUPERF_UTILE_RISCALDATA           range=[3.39, 84477.85]  median=67.64
  VOLUME_LORDO_RISCALDATO           range=[20.71, 3460005.00]  median=264.91
  RAPPORTO_SV                       range=[0.00, 2.73]  median=0.39
  SUPERFICIE_DISPERDENTE            range=[5.00, 135883.92]  median=104.85
  NUMERO_UNITA_IMMOBILIARI          range=[1.00, 1732.00]  median=1.00
  A_SOL_EST_A_SUP_UTILE             range=[0.00, 0.44]  median=0.03
  CDD26                             range=[22.40, 22.40]  median=22.40
  mean_temp                         range=[13.11, 13.11]  median=13.11
  mean_temp_summer                  range=[22.93, 22.93]  median=22.93
  max_temp                          ran

In [8]:
# =============================================================
# STEP 4 — CELL 13b
# Investigate outliers and apply fixes before final assembly
# =============================================================

print("Investigating outliers and applying fixes...\n")

# =============================================================
# INVESTIGATION 1 — construction_year < 1900
# =============================================================
print("[1] Investigating buildings with construction_year < 1900:")
old_buildings = working_df[working_df['construction_year'] < 1900]
print(f"    Total: {len(old_buildings):,}")
print(f"    Construction year distribution (< 1900):")
print(old_buildings['construction_year'].value_counts().sort_index().to_string())
print(f"    Original ANNO_COSTRUZIONE values for these buildings:")
print(old_buildings.merge(df[['ANNO_COSTRUZIONE']], left_index=True, right_index=True)
      ['ANNO_COSTRUZIONE'].value_counts().to_string())

# =============================================================
# FIX 1 — Update reference year to 2025
# =============================================================
print("\n[2] Updating reference year from 2024 to 2025 (latest data is 2025)")
working_df['building_age'] = 2025 - working_df['construction_year']
print(f"    building_age min: {working_df['building_age'].min():.0f}")
print(f"    building_age max: {working_df['building_age'].max():.0f}")
print(f"    Negative ages remaining: {(working_df['building_age'] < 0).sum()}")

# =============================================================
# FIX 2 — Convert categorical features to string type
# =============================================================
print("\n[3] Converting categorical codes to proper string type:")
for col in ['EDIFC_TY', 'EDIFC_USO', 'EDIFC_STAT']:
    working_df[col] = working_df[col].astype(str)
    print(f"    {col}: converted to string ({working_df[col].nunique()} unique values)")

# =============================================================
# FIX 3 — Decide on the 1500 outliers
# =============================================================
print("\n[4] Decision on 1500-year outliers:")
n_1500 = (working_df['construction_year'] <= 1500).sum()
print(f"    Buildings with construction_year <= 1500: {n_1500}")
print(f"    These are clearly data-entry errors (Milan as a city is older but few buildings survive).")
print(f"    Recommendation: Will replace with median of 'Prima del 1930' bucket = 1920")

# Apply the fix
mask_1500 = working_df['construction_year'] <= 1500
n_fixed = mask_1500.sum()
working_df.loc[mask_1500, 'construction_year'] = 1920
working_df.loc[mask_1500, 'building_age']      = 2025 - 1920
print(f"    Replaced {n_fixed} extreme outliers (set to year 1920, age 105).")

# =============================================================
# RE-VERIFY THE NUMERIC RANGES
# =============================================================
print("\n" + "="*70)
print("UPDATED FEATURE RANGES (after fixes)")
print("="*70)

key_numeric = ['construction_year', 'building_age',
               'SUPERF_UTILE_RISCALDATA', 'VOLUME_LORDO_RISCALDATO',
               'NUMERO_UNITA_IMMOBILIARI']
for f in key_numeric:
    s = working_df[f]
    print(f"  {f:<32}  range=[{s.min():.2f}, {s.max():.2f}]  median={s.median():.2f}")

print("\nDone. Ready for Cell 14 (final encoding & save).")

Investigating outliers and applying fixes...

[1] Investigating buildings with construction_year < 1900:
    Total: 56
    Construction year distribution (< 1900):
construction_year
1500.0     2
1700.0     4
1712.0     1
1750.0     1
1800.0     8
1850.0     6
1855.0     1
1860.0     2
1865.0     1
1870.0     2
1880.0     9
1886.0     3
1887.0     1
1889.0     1
1890.0    13
1894.0     1
    Original ANNO_COSTRUZIONE values for these buildings:
ANNO_COSTRUZIONE
1890    13
1880     9
1800     8
1850     6
1700     4
1886     3
1870     2
1500     2
1860     2
1750     1
1887     1
1855     1
1889     1
1894     1
1865     1
1712     1

[2] Updating reference year from 2024 to 2025 (latest data is 2025)
    building_age min: 0
    building_age max: 525
    Negative ages remaining: 0

[3] Converting categorical codes to proper string type:
    EDIFC_TY: converted to string (6 unique values)
    EDIFC_USO: converted to string (21 unique values)
    EDIFC_STAT: converted to string (3 unique 

In [9]:
# =============================================================
# STEP 4 — CELL 14
# Final encoding: log-transform, one-hot, boolean conversion
# =============================================================

print("Building the final feature matrix X and target vector y...\n")

# ---- WORK ON A COPY ----
encoded_df = working_df.copy()

# =============================================================
# TRANSFORMATION 1 — Log-transform skewed size variables
# =============================================================
print("[1] Adding log-transformed versions of skewed size variables")
print("    (Originals are kept for tree-based models; logs are for linear models)")

skewed_features = ['SUPERF_UTILE_RISCALDATA', 'VOLUME_LORDO_RISCALDATO',
                   'SUPERFICIE_DISPERDENTE', 'NUMERO_UNITA_IMMOBILIARI']
for f in skewed_features:
    log_name = f'log_{f}'
    # log1p handles zero values safely: log(1+x)
    encoded_df[log_name] = np.log1p(encoded_df[f])
    print(f"    Created {log_name}  (range: [{encoded_df[log_name].min():.2f}, {encoded_df[log_name].max():.2f}])")

# =============================================================
# TRANSFORMATION 2 — Convert booleans to integers (0/1)
# =============================================================
print("\n[2] Converting boolean features to integers (0/1)")
for f in boolean_features:
    encoded_df[f] = encoded_df[f].astype(int)
    print(f"    {f:<35} 0={(encoded_df[f]==0).sum():>6,}  1={(encoded_df[f]==1).sum():>6,}")

# Convert target to integer (0/1)
encoded_df[target_col] = encoded_df[target_col].astype(int)

# =============================================================
# TRANSFORMATION 3 — One-hot encoding of categorical features
# =============================================================
print("\n[3] One-hot encoding categorical features")
print("    (Each unique category becomes its own 0/1 column)")

# Use drop_first=False to keep all categories — better interpretability for SHAP later
shape_before = encoded_df.shape
encoded_df = pd.get_dummies(encoded_df, columns=categorical_features, drop_first=False, dtype=int)
shape_after = encoded_df.shape

new_cols = shape_after[1] - shape_before[1] + len(categorical_features)
print(f"    Categorical features encoded: {len(categorical_features)}")
print(f"    New dummy columns created:    {new_cols}")
print(f"    Shape: {shape_before[1]} columns -> {shape_after[1]} columns")

# Show the new dummy columns
dummy_cols = [c for c in encoded_df.columns
              if any(c.startswith(cat + '_') for cat in categorical_features)]
print(f"\n    Sample of new dummy columns ({min(8, len(dummy_cols))} of {len(dummy_cols)}):")
for c in dummy_cols[:8]:
    print(f"      {c}")

# =============================================================
# TRANSFORMATION 4 — Build final X and y
# =============================================================
print("\n[4] Building final X (features) and y (target)")

y = encoded_df[target_col]
X = encoded_df.drop(columns=[target_col])

print(f"    X shape: {X.shape[0]:,} rows × {X.shape[1]} columns")
print(f"    y shape: {y.shape[0]:,} rows")
print(f"    All values in X are numeric: {X.dtypes.apply(lambda d: pd.api.types.is_numeric_dtype(d)).all()}")
print(f"    Missing values in X: {X.isnull().sum().sum()}")
print(f"    Missing values in y: {y.isnull().sum()}")

# =============================================================
# FINAL SUMMARY
# =============================================================
print("\n" + "="*70)
print("FINAL ML-READY DATASET")
print("="*70)
print(f"  Buildings (rows):     {X.shape[0]:,}")
print(f"  Features (columns):   {X.shape[1]}")
print(f"  Target distribution:")
print(f"    Cooling YES (1): {(y==1).sum():>6,}  ({(y==1).mean()*100:.1f}%)")
print(f"    Cooling NO  (0): {(y==0).sum():>6,}  ({(y==0).mean()*100:.1f}%)")
print(f"\n  Feature breakdown:")
print(f"    Numeric (incl. log-transforms): ~{len(numeric_features) + len(skewed_features)}")
print(f"    Boolean (now 0/1):              {len(boolean_features)}")
print(f"    One-hot encoded categories:     {len(dummy_cols)}")

print("\nDone. X and y are ready for ML modelling.")

Building the final feature matrix X and target vector y...

[1] Adding log-transformed versions of skewed size variables
    (Originals are kept for tree-based models; logs are for linear models)
    Created log_SUPERF_UTILE_RISCALDATA  (range: [1.48, 11.34])
    Created log_VOLUME_LORDO_RISCALDATO  (range: [3.08, 15.06])
    Created log_SUPERFICIE_DISPERDENTE  (range: [1.79, 11.82])
    Created log_NUMERO_UNITA_IMMOBILIARI  (range: [0.69, 7.46])

[2] Converting boolean features to integers (0/1)
    INTERO_EDIFICIO                     0=18,714  1=   349
    NUOVA_COSTRUZIONE                   0=18,868  1=   195
    RISTRUTTURAZIONE_IMPORTANTE         0=18,676  1=   387
    RIQUALIFICAZIONE_ENERGETICA         0=18,404  1=   659
    PASSAGGIO_PROPRIETA                 0= 8,368  1=10,695
    LOCAZIONE                           0=13,624  1= 5,439

[3] One-hot encoding categorical features
    (Each unique category becomes its own 0/1 column)
    Categorical features encoded: 4
    New dum

In [10]:
# =============================================================
# STEP 4 — CELL 15
# Save final ML-ready dataset and full transformation log
# =============================================================

from datetime import datetime

print("Saving the final ML-ready dataset and traceability log...\n")

# ---- 1. SAVE THE COMPLETE ML-READY DATASET (X + y combined) ----
final_dataset = X.copy()
final_dataset['CLIMATIZZAZIONE_ESTIVA'] = y

output_path = FINAL_DIR / "stage1_ml_features_v1.csv"
print(f"[1] Saving complete dataset (X + y): {output_path.name}")
final_dataset.to_csv(output_path, index=False)
size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"    [OK] {final_dataset.shape[0]:,} rows × {final_dataset.shape[1]} columns ({size_mb:.2f} MB)")

# ---- 2. SAVE FEATURES-ONLY (X) AND TARGET-ONLY (y) ----
X_path = FINAL_DIR / "stage1_X_features_v1.csv"
y_path = FINAL_DIR / "stage1_y_target_v1.csv"

X.to_csv(X_path, index=False)
y.to_frame(name='CLIMATIZZAZIONE_ESTIVA').to_csv(y_path, index=False)

print(f"\n[2] Saved separate files for convenience:")
print(f"    - {X_path.name}  (features only)")
print(f"    - {y_path.name}  (target only)")

# ---- 3. SAVE FEATURE METADATA ----
metadata_rows = []
for col in X.columns:
    s = X[col]
    metadata_rows.append({
        'feature_name': col,
        'dtype':        str(s.dtype),
        'min':          float(s.min()),
        'max':          float(s.max()),
        'mean':         float(s.mean()),
        'std':          float(s.std()),
        'n_unique':     int(s.nunique()),
        'n_missing':    int(s.isnull().sum()),
    })
metadata_df = pd.DataFrame(metadata_rows)
metadata_path = LOGS_DIR / "step4_feature_metadata.csv"
metadata_df.to_csv(metadata_path, index=False)
print(f"\n[3] Feature metadata saved: 05_logs/{metadata_path.name}")
print(f"    ({len(metadata_df)} features documented)")

# ---- 4. SAVE STEP 4 README ----
readme_path = LOGS_DIR / "step4_README.txt"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("STEP 4 — FEATURE ENGINEERING AND SELECTION\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Date generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

    f.write("PURPOSE:\n")
    f.write("  Transform the 19,063 × 141 cleaned dataset into a fully numeric,\n")
    f.write("  ML-ready feature matrix for Stage 1 binary classification.\n\n")

    f.write("INPUT:\n")
    f.write("  04_final_ml_dataset/stage1_ml_dataset_v1.csv (19,063 × 141)\n\n")

    f.write("OUTPUT:\n")

Saving the final ML-ready dataset and traceability log...

[1] Saving complete dataset (X + y): stage1_ml_features_v1.csv
    [OK] 19,063 rows × 62 columns (6.50 MB)

[2] Saved separate files for convenience:
    - stage1_X_features_v1.csv  (features only)
    - stage1_y_target_v1.csv  (target only)

[3] Feature metadata saved: 05_logs/step4_feature_metadata.csv
    (61 features documented)


In [11]:
# =============================================================
# STEP 4 — CELL 15
# Save final ML-ready dataset and full transformation log
# =============================================================

from datetime import datetime

print("Saving the final ML-ready dataset and traceability log...\n")

# ---- 1. SAVE THE COMPLETE ML-READY DATASET (X + y combined) ----
final_dataset = X.copy()
final_dataset['CLIMATIZZAZIONE_ESTIVA'] = y

output_path = FINAL_DIR / "stage1_ml_features_v1.csv"
print(f"[1] Saving complete dataset (X + y): {output_path.name}")
final_dataset.to_csv(output_path, index=False)
size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"    [OK] {final_dataset.shape[0]:,} rows × {final_dataset.shape[1]} columns ({size_mb:.2f} MB)")

# ---- 2. SAVE FEATURES-ONLY (X) AND TARGET-ONLY (y) ----
X_path = FINAL_DIR / "stage1_X_features_v1.csv"
y_path = FINAL_DIR / "stage1_y_target_v1.csv"

X.to_csv(X_path, index=False)
y.to_frame(name='CLIMATIZZAZIONE_ESTIVA').to_csv(y_path, index=False)

print(f"\n[2] Saved separate files for convenience:")
print(f"    - {X_path.name}  (features only)")
print(f"    - {y_path.name}  (target only)")

# ---- 3. SAVE FEATURE METADATA ----
metadata_rows = []
for col in X.columns:
    s = X[col]
    metadata_rows.append({
        'feature_name': col,
        'dtype':        str(s.dtype),
        'min':          float(s.min()),
        'max':          float(s.max()),
        'mean':         float(s.mean()),
        'std':          float(s.std()),
        'n_unique':     int(s.nunique()),
        'n_missing':    int(s.isnull().sum()),
    })
metadata_df = pd.DataFrame(metadata_rows)
metadata_path = LOGS_DIR / "step4_feature_metadata.csv"
metadata_df.to_csv(metadata_path, index=False)
print(f"\n[3] Feature metadata saved: 05_logs/{metadata_path.name}")
print(f"    ({len(metadata_df)} features documented)")

# ---- 4. SAVE STEP 4 README ----
readme_path = LOGS_DIR / "step4_README.txt"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("STEP 4 — FEATURE ENGINEERING AND SELECTION\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Date generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

    f.write("PURPOSE:\n")
    f.write("  Transform the 19,063 × 141 cleaned dataset into a fully numeric,\n")
    f.write("  ML-ready feature matrix for Stage 1 binary classification.\n\n")

    f.write("INPUT:\n")
    f.write("  04_final_ml_dataset/stage1_ml_dataset_v1.csv (19,063 × 141)\n\n")

    f.write("OUTPUT:\n")
    f.write("  04_final_ml_dataset/stage1_ml_features_v1.csv (19,063 × 62)\n")
    f.write("  04_final_ml_dataset/stage1_X_features_v1.csv  (19,063 × 61, X only)\n")
    f.write("  04_final_ml_dataset/stage1_y_target_v1.csv    (19,063 × 1, y only)\n")
    f.write("  05_logs/step4_feature_metadata.csv\n\n")

    f.write("APPROACH:\n")
    f.write("  Conservative feature selection based on domain knowledge.\n")
    f.write("  Started from 26 candidate features in 4 logical groups\n")
    f.write("  (DBT geometry, EPC building info, design, climate).\n\n")

    f.write("TRANSFORMATIONS APPLIED:\n\n")

    f.write("  T1. EDIFC_CASS removed from feature list\n")
    f.write("      Reason: 19,063 unique values across 19,063 rows = unique identifier\n\n")

    f.write("  T2. ANNO_COSTRUZIONE converted from text to numeric\n")
    f.write("      Method: midpoint of decade ranges (e.g., '1961-1976' -> 1968)\n")
    f.write("      Special cases:\n")
    f.write("        - 'Prima del 1930' (2,532 buildings) -> 1920\n")
    f.write("        - 'Dopo il 2006'    (412 buildings) -> 2015\n")
    f.write("      Parse rate: 100% (19,063 / 19,063)\n\n")

    f.write("  T3. building_age derived feature created\n")
    f.write("      Formula: 2025 - construction_year\n")
    f.write("      Range: [0, 325] years\n\n")

    f.write("  T4. Two extreme outliers fixed\n")
    f.write("      2 buildings with construction_year = 1500 reset to 1920\n")
    f.write("      Reason: clearly data-entry errors\n\n")

    f.write("  T5. Log-transforms added for skewed size variables\n")
    f.write("      log1p applied to: SUPERF_UTILE_RISCALDATA, VOLUME_LORDO_RISCALDATO,\n")
    f.write("      SUPERFICIE_DISPERDENTE, NUMERO_UNITA_IMMOBILIARI\n")
    f.write("      Originals retained for tree-based models;\n")
    f.write("      logs added for linear-model robustness.\n\n")

    f.write("  T6. Boolean features converted to integers (0/1)\n")
    f.write("      6 features: INTERO_EDIFICIO, NUOVA_COSTRUZIONE, RISTRUTTURAZIONE_IMPORTANTE,\n")
    f.write("      RIQUALIFICAZIONE_ENERGETICA, PASSAGGIO_PROPRIETA, LOCAZIONE\n\n")

    f.write("  T7. One-hot encoding of categorical features\n")
    f.write("      4 features expanded into 35 dummy columns:\n")
    f.write("        - EDIFC_TY (6 categories)\n")
    f.write("        - EDIFC_USO (21 categories)\n")
    f.write("        - EDIFC_STAT (3 categories)\n")
    f.write("        - CLASSIFICAZIONE_DPR (5 categories)\n\n")

    f.write("FINAL DATASET CHARACTERISTICS:\n")
    f.write(f"  - 19,063 buildings (no rows lost)\n")
    f.write(f"  - 61 features (numeric, ready for any ML algorithm)\n")
    f.write(f"  - 0 missing values\n")
    f.write(f"  - Class balance: 72.3% YES / 27.7% NO\n")
    f.write(f"  - Imbalance ratio: 2.61 : 1 (manageable)\n\n")

    f.write("KEY DESIGN DECISIONS (for thesis defence):\n")
    f.write("  1. Conservative feature set chosen over aggressive: 26 candidates\n")
    f.write("     selected from 141 cleaned columns based on domain knowledge.\n")
    f.write("  2. Climate features kept despite current constancy:\n")
    f.write("     essential for Stage 2 future-scenario predictions.\n")
    f.write("  3. Both raw and log-transformed size variables retained:\n")
    f.write("     allows model-specific selection (linear vs tree-based).\n")
    f.write("  4. All categorical encoding uses drop_first=False:\n")
    f.write("     improves SHAP interpretability in later steps.\n\n")

    f.write("NEXT STEPS:\n")
    f.write("  Step 5: Train/test split, baseline modelling (Logistic Regression)\n")
    f.write("  Step 6: Advanced models (Random Forest, XGBoost) and tuning\n")
    f.write("  Step 7: Model evaluation (ROC-AUC, F1, calibration, confusion matrix)\n")
    f.write("  Step 8: SHAP / LIME explainability analysis\n")
    f.write("  Step 9: Predict on the 33,978 unlabelled buildings (full city inference)\n")
    f.write("  Step 10: Stage 2 — cooling intensity regression, future climate scenarios\n")
print(f"\n[4] Step 4 README saved: 05_logs/{readme_path.name}")

# ---- FINAL SUMMARY ----
print("\n" + "="*70)
print("STEP 4 — COMPLETE")
print("="*70)
print(f"  Input shape:  19,063 × 141")
print(f"  Output shape: 19,063 × 61 features (+ 1 target)")
print(f"  Files written:")
print(f"    1. 04_final_ml_dataset/stage1_ml_features_v1.csv  (X + y combined)")
print(f"    2. 04_final_ml_dataset/stage1_X_features_v1.csv    (X only)")
print(f"    3. 04_final_ml_dataset/stage1_y_target_v1.csv      (y only)")
print(f"    4. 05_logs/step4_feature_metadata.csv")
print(f"    5. 05_logs/step4_README.txt")
print(f"\n  Ready for Step 5 (model training).")
print("\nDone.")

Saving the final ML-ready dataset and traceability log...

[1] Saving complete dataset (X + y): stage1_ml_features_v1.csv
    [OK] 19,063 rows × 62 columns (6.50 MB)

[2] Saved separate files for convenience:
    - stage1_X_features_v1.csv  (features only)
    - stage1_y_target_v1.csv  (target only)

[3] Feature metadata saved: 05_logs/step4_feature_metadata.csv
    (61 features documented)

[4] Step 4 README saved: 05_logs/step4_README.txt

STEP 4 — COMPLETE
  Input shape:  19,063 × 141
  Output shape: 19,063 × 61 features (+ 1 target)
  Files written:
    1. 04_final_ml_dataset/stage1_ml_features_v1.csv  (X + y combined)
    2. 04_final_ml_dataset/stage1_X_features_v1.csv    (X only)
    3. 04_final_ml_dataset/stage1_y_target_v1.csv      (y only)
    4. 05_logs/step4_feature_metadata.csv
    5. 05_logs/step4_README.txt

  Ready for Step 5 (model training).

Done.
